#Data Ingestion for Qdrant Hybrid Search


This script ingests California housing statutes into Qdrant with:
- Dense vectors (semantic embeddings via BGE-small)
- Sparse vectors (keyword matching via BM42)
- Hybrid search ready for RAG demos

In [ ]:
#pip install qdrant-client[fastembed] "datasets==3.6.0" "huggingface_hub<0.27.0"

In [ ]:
import os
import sys
import time
from datasets import load_dataset
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, SparseIndexParams
from fastembed import TextEmbedding, SparseTextEmbedding
from google.colab import userdata

# Configuration
QDRANT_URL = userdata.get('QDRANT_CLOUD_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
COLLECTION_NAME = "california_housing_statutes"

# Embedding models
DENSE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL_NAME = "Qdrant/bm42-all-minilm-l6-v2-attentions"

# Vector configuration
DENSE_VECTOR_SIZE = 384
DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse"

# Batching
BATCH_SIZE = 32
RATE_LIMIT_DELAY = 0.3  # seconds between batches

print("✓ Configuration loaded")

✓ Configuration loaded


## Configuration Options

Set these variables to control the ingestion process:

In [ ]:
# Set ingestion parameters
LIMIT_STATUTES = 100  # Set to a number (e.g., 100) to test with fewer statutes
RECREATE_COLLECTION = False  # Set to True to delete existing collection and start fresh

## Step 1: Check Environment


## Step 2: Initialize Clients


In [ ]:
# Initialize Qdrant client
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Load embedding models
dense_model = TextEmbedding(model_name=DENSE_MODEL_NAME)

sparse_model = SparseTextEmbedding(model_name=SPARSE_MODEL_NAME)

## Step 3: Load California Statutes


In [ ]:
print("STEP 1: Getting the data")
print("-" * 80)

print("Downloading statutes from Huggingface...")

# Load full corpus-this will take a few minutes as we are loading 1,837,403 statues and filtering for California
dataset = load_dataset(
    "reglab/housing_qa",
    "statutes",
    split="corpus",
    trust_remote_code=True
)

# Filter for California
california_statutes = []
for statute in dataset:
    if statute.get('state', '').lower() == 'california':
        california_statutes.append({
            'idx': statute['idx'],
            'citation': statute['citation'],
            'state': statute['state'],
            'text': statute['text'],
            'path': statute.get('path', '')
        })


        # Stop when limit reached
        if LIMIT_STATUTES and len(california_statutes) >= LIMIT_STATUTES:
            print(f"  → Fetched {LIMIT_STATUTES} statutes from California")
            break

STEP 1: Getting the data
--------------------------------------------------------------------------------
  → Fetched 100 statutes from California


## Step 4: Create or Check Qdrant Collection


In [ ]:
print("STEP 2: Setting Up Qdrant Collection")
print("-" * 80)

# Check if collection exists
collections = client.get_collections().collections
exists = any(c.name == COLLECTION_NAME for c in collections)

if exists:
    if RECREATE_COLLECTION:
        print(f"  → Collection '{COLLECTION_NAME}' exists. Deleting...")
        client.delete_collection(COLLECTION_NAME)
        print("  → Deleted")
        exists = False
    else:
        print(f"  → Collection '{COLLECTION_NAME}' already exists")
        info = client.get_collection(COLLECTION_NAME)
        print(f"  → Current count: {info.points_count:,} points")
        print("  → Will append to existing collection\n")

if not exists:
    # Create new collection with dense + sparse vectors
    print(f"  → Creating collection: {COLLECTION_NAME}")
    print(f"  → Dense vector: {DENSE_MODEL_NAME} ({DENSE_VECTOR_SIZE}D)")
    print(f"  → Sparse vector: {SPARSE_MODEL_NAME}")

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            DENSE_VECTOR_NAME: VectorParams(
                size=DENSE_VECTOR_SIZE,
                distance=Distance.COSINE
            )
        },
        sparse_vectors_config={
            SPARSE_VECTOR_NAME: SparseVectorParams(
                index=SparseIndexParams(
                    on_disk=False,
                )
            )
        }
    )

    print(f"✓ Collection created: {COLLECTION_NAME}\n")


STEP 2: Setting Up Qdrant Collection
--------------------------------------------------------------------------------
  → Creating collection: california_housing_statutes
  → Dense vector: BAAI/bge-small-en-v1.5 (384D)
  → Sparse vector: Qdrant/bm42-all-minilm-l6-v2-attentions
✓ Collection created: california_housing_statutes



## Step 5: Embed and Upload Statutes


In [ ]:
print("STEP 3: Embedding and Uploading Statutes")
print("-" * 80)

total_batches = (len(california_statutes) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"  → Processing {len(california_statutes):,} statutes in {total_batches} batches")
print(f"  → Batch size: {BATCH_SIZE}")
print()

start_time = time.time()

# Get starting point ID (in case collection already has points)
try:
    info = client.get_collection(COLLECTION_NAME)
    start_id = info.points_count
except:
    start_id = 0

for batch_idx in range(0, len(california_statutes), BATCH_SIZE):
    batch_num = batch_idx // BATCH_SIZE + 1
    batch = california_statutes[batch_idx:batch_idx + BATCH_SIZE]

    # Extract texts
    texts = [s['text'] for s in batch]

    # Generate dense embeddings
    dense_embeddings = list(dense_model.embed(texts))

    # Generate sparse embeddings
    sparse_embeddings = list(sparse_model.embed(texts))

    # Create points
    points = []
    for i, statute in enumerate(batch):
        point_id = start_id + batch_idx + i

        # Convert sparse embedding to Qdrant format
        sparse_values = models.SparseVector(
            indices=sparse_embeddings[i].indices.tolist(),
            values=sparse_embeddings[i].values.tolist()
        )

        points.append(
            models.PointStruct(
                id=point_id,
                vector={
                    DENSE_VECTOR_NAME: dense_embeddings[i].tolist(),
                    SPARSE_VECTOR_NAME: sparse_values
                },
                payload={
                    'statute_idx': statute['idx'],
                    'citation': statute['citation'],
                    'state': statute['state'],
                    'text': statute['text'],
                    'path': statute['path']
                }
            )
        )

    # Upload batch
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )

    # Progress
    progress = (batch_num / total_batches) * 100
    print(f"  [{batch_num}/{total_batches}] {progress:.1f}% - Uploaded {len(points)} statutes")

    # Rate limiting
    time.sleep(RATE_LIMIT_DELAY)

elapsed = time.time() - start_time
print(f"\n✓ Uploaded {len(california_statutes):,} statutes in {elapsed:.1f}s\n")


STEP 3: Embedding and Uploading Statutes
--------------------------------------------------------------------------------
  → Processing 100 statutes in 4 batches
  → Batch size: 32

  [1/4] 25.0% - Uploaded 32 statutes
  [2/4] 50.0% - Uploaded 32 statutes
  [3/4] 75.0% - Uploaded 32 statutes
  [4/4] 100.0% - Uploaded 4 statutes

✓ Uploaded 100 statutes in 56.2s



In [ ]:
print("STEP 4: Verifying Collection")
print("-" * 80)

# Get collection info
info = client.get_collection(COLLECTION_NAME)

print(f"  Collection: {COLLECTION_NAME}")
print(f"  Point count: {info.points_count:,}")
print(f"  Vectors:")
print(f"    • Dense ({DENSE_VECTOR_NAME}): {DENSE_VECTOR_SIZE}D")
print(f"    • Sparse ({SPARSE_VECTOR_NAME}): Active")

# Test search
print("\n  → Running test search...")

test_query = "Can a landlord evict a tenant?"

# Generate test embeddings
dense_query = list(dense_model.embed([test_query]))[0].tolist()
sparse_query_embedding = list(sparse_model.embed([test_query]))[0]
sparse_query = models.SparseVector(
    indices=sparse_query_embedding.indices.tolist(),
    values=sparse_query_embedding.values.tolist()
)

# Hybrid search
results = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=dense_query, using=DENSE_VECTOR_NAME, limit=10),
        models.Prefetch(query=sparse_query, using=SPARSE_VECTOR_NAME, limit=10),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3,
)

print(f"  → Retrieved {len(results.points)} results")

if results.points:
    print("\n  Top result:")
    top = results.points[0]
    print(f"    Citation: {top.payload.get('citation', 'Unknown')}")
    print(f"    Score: {top.score:.3f}")
    print(f"    Text: {top.payload.get('text', '')[:100]}...")

print("\n✓ Collection verified and ready!\n")


STEP 4: Verifying Collection
--------------------------------------------------------------------------------
  Collection: california_housing_statutes
  Point count: 100
  Vectors:
    • Dense (dense): 384D
    • Sparse (sparse): Active

  → Running test search...
  → Retrieved 3 results

  Top result:
    Citation: CA Civ Pro Code § 6 (2021)
    Score: 0.500
    Text: 6.  All persons who at the time this Code takes effect hold office under any of the Acts repealed, c...

✓ Collection verified and ready!



## ✅ Ingestion Complete!


In [ ]:
print("="*80)
print("INGESTION COMPLETE!")
print("="*80)
print(f"""
✅ Successfully ingested {len(california_statutes):,} California housing statutes

Collection Details:
  • Name: {COLLECTION_NAME}
  • Dense vectors: {DENSE_MODEL_NAME}
  • Sparse vectors: {SPARSE_MODEL_NAME}
  • Hybrid search: RRF fusion enabled

Next Steps:
  1. Run vanilla RAG demo: demo_vanilla_rag.ipynb
  2. Run hybrid search demo: demo_qdrant_hybrid_search.ipynb
  3. Run agentic RAG demo: demo_crewai_qdrant_agent.ipynb

Ready for rumble! 🚀
""")


INGESTION COMPLETE!

✅ Successfully ingested 100 California housing statutes

Collection Details:
  • Name: california_housing_statutes
  • Dense vectors: BAAI/bge-small-en-v1.5
  • Sparse vectors: Qdrant/bm42-all-minilm-l6-v2-attentions
  • Hybrid search: RRF fusion enabled

Next Steps:
  1. Run vanilla RAG demo: demo_vanilla_rag.ipynb
  2. Run hybrid search demo: demo_qdrant_hybrid_search.ipynb
  3. Run agentic RAG demo: demo_crewai_qdrant_agent.ipynb

Ready for rumble! 🚀

